In [113]:
import pymupdf
import numpy as np

def extract_text_pymupdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    full_text = ""
    
    for page in doc:
        full_text += page.get_text("text")
        
    doc.close()
    return full_text

def process(text):
    text = text.split("\n")
    text = [k.split(":") for k in text if len(k.split(":")) == 2]
    return text

extracted = extract_text_pymupdf("demo.pdf")
params = process(extracted)
params = np.array(params)
params = [[p[0] for p in params], [p[1] for p in params]]
params = dict(zip([p.strip() for p in params[0]], [p.strip() for p in params[1]]))

params

{'Patient ID': 'TG-FZUFCRZS3',
 'Site ID': 'SITE-TMP-01',
 'Triage Nurse ID': 'NURSE-0050',
 'Arrival Mode': 'walk-in',
 'Arrival Hour': '7',
 'Arrival Day': 'Monday',
 'Arrival Month': '4',
 'Arrival Season': 'spring',
 'Shift': 'morning',
 'Age': '24',
 'Age Group': 'young_adult',
 'Sex': 'M',
 'Language': 'English',
 'Insurance Type': 'public',
 'Transport Origin': 'public_space',
 'Prior ED Visits (12m)': '2',
 'Prior Admissions (12m)': '2',
 'Active Medications': '4',
 'Comorbidities': '4',
 'Pain Location': 'none',
 'Mental Status (Triage)': 'confused',
 'Chief Complaint System': 'musculoskeletal',
 'Systolic BP': '73.0',
 'Diastolic BP': '77.9',
 'Mean Arterial Pressure': '76.3',
 'Pulse Pressure': '-4.9',
 'Heart Rate': '75.3',
 'Respiratory Rate': '26.7',
 'Temperature (°C)': '38.5',
 'SpO2': '88.8',
 'GCS Total': '10',
 'Pain Score': '10',
 'Weight (kg)': '86.9',
 'Height (cm)': '177.2',
 'BMI': '27.7',
 'Shock Index': '1.032',
 'NEWS2 Score': '13'}

In [114]:
# Process fields into reqd fields

import csv

f = open("model.csv", 'r')

reader = list(csv.reader(f.readlines()))
header = reader[0]
weights = reader[1]
intercept = float(reader[2][0])

fields = dict()
for i, h in enumerate(header):
    fields[h] = float(weights[i])

fields

{'arrival_mode': 0.001325769307131233,
 'age_group': -0.005314758366319801,
 'age': -1.6605643438825475e-06,
 'num_prior_ed_visits_12m': 0.1257339657220237,
 'num_prior_admissions_12m': -0.0074174571486471386,
 'num_active_medications': 0.0001607885017735725,
 'num_comorbidities': 0.00022150971648483972,
 'systolic_bp': 6.091626949328199e-05,
 'diastolic_bp': 0.0004389954330244497,
 'mean_arterial_pressure': -0.0005314796352848906,
 'pulse_pressure': 3.2973311062823604e-09,
 'heart_rate': -2.3629554821677703e-09,
 'respiratory_rate': -2.0636271669141686e-07,
 'temperature_c': 0.0025212006386802813,
 'spo2': -5.24236405091187e-08,
 'gcs_total': 2.4831644753848334e-05,
 'pain_score': 0.06634725190450828,
 'bmi': 1.1776306607293347e-08,
 'shock_index': 0.037766820444874734,
 'news2_score': 0.13714166084207668}

In [115]:
import pandas as pd
from typing import List

text_to_field = {
 'arrival_mode': 'Arrival Mode',
 'age_group': 'Age Group',
 'age': 'Age',
 'num_prior_ed_visits_12m': 'Prior ED Visits (12m)',
 'num_prior_admissions_12m': 'Prior Admissions (12m)',
 'num_active_medications': 'Active Medications',
 'num_comorbidities': 'Comorbidities',
 'systolic_bp': 'Systolic BP',
 'diastolic_bp': 'Diastolic BP',
 'mean_arterial_pressure': 'Mean Arterial Pressure',
 'pulse_pressure': 'Pulse Pressure',
 'heart_rate': 'Heart Rate',
 'respiratory_rate': 'Respiratory Rate',
 'temperature_c': 'Temperature (°C)',
 'spo2': 'SpO2',
 'gcs_total': 'GCS Total',
 'pain_score': 'Pain Score',
 'bmi': 'BMI',
 'shock_index': 'Shock Index',
 'news2_score': 'NEWS2 Score'
}

defaults = {
 'arrival_mode': 0,
 'age_group': 1,
 'age': 40,
 'num_prior_ed_visits_12m': 0,
 'num_prior_admissions_12m': 0,
 'num_active_medications': 0,
 'num_comorbidities': 0,
 'systolic_bp': 120,
 'diastolic_bp': 90,
 'mean_arterial_pressure': 80,
 'pulse_pressure': 40,
 'heart_rate': 72,
 'respiratory_rate': 16,
 'temperature_c': 37,
 'spo2': 100,
 'gcs_total': 15,
 'pain_score': 0,
 'bmi': 21,
 'shock_index': 0.6,
 'news2_score': 0
}

values = {}

for field in fields:
    if text_to_field[field] in params:
        values[field] = params[text_to_field[field]]

# Define predefined functions

values = pd.DataFrame(values, index=[0])

def systolic_blood_pressure(s):
    if 90 <= s <= 120:
        return 0
    if s < 90:
        return (s - 90) ** 2
    if 140 >= s > 120:
        return abs(s - 120) ** 1.5
    else:
        return 89.44 + ((s - 140) ** 2)

def diastolic_blood_pressure(s):
    if 60 <= s <= 80:
        return 0
    if s < 60:
        return (s - 60) ** 2
    if 90 >= s > 80:
        return abs(s - 80) ** 1.5
    else:
        return 31.62 + ((s - 90) ** 2)

def map(s):
    if 70 <= s <= 100:
        return 0
    if s < 70:
        return (s - 70) ** 2
    if 100 <= s < 110:
        return abs(s - 100)
    else:
        return abs(s - 110) ** 2

def pulse_pressure(p):
    return (p - 40) ** 4

def respiratory_rate(age: List[float], respiratory_rate: List[float]) -> List[float]:
    res = [0.0] * len(age)
    for i in range(len(age)):
        if age[i] < 1:
            res[i] = abs(respiratory_rate[i] - 45) ** 4
            continue
        elif age[i] < 2:
            res[i] = abs(respiratory_rate[i] - 30) ** 4
            continue
        elif 3 < age[i] < 6:
            res[i] = abs(respiratory_rate[i] - 25) ** 4
            continue
        elif 6 < age[i] < 12:
            res[i] = abs(respiratory_rate[i] - 22.5) ** 4
            continue
        else:
            res[i] = abs(respiratory_rate[i] - 16) ** 4
            continue
    return res

def temp(t):
    if 36 < t < 38:
        return 0
    if t > 38:
        return (t - 38) ** 4
    return abs(36 - t) ** 4

def bmi(b):
    if 18.5 <= b <= 25:
        return 0
    if b < 18.5:
        return (b - 18.5) ** 4
    else:
        return (25 - b) ** 4

calculated = pd.DataFrame()
calculated['arrival_mode'] = values['arrival_mode'].apply(lambda x: 1 if x in ["ambulance", 'transfer', 'police', 'helicoper'] else 0)
calculated['age_group'] = values['age_group'].apply(lambda x: 2 if x in ['elderly', 'pediatric'] else 1)
calculated['age'] = values['age'].astype(int).apply(lambda x: abs(x - 35) ** 2)
calculated['num_prior_ed_visits_12m'] = values['num_prior_ed_visits_12m'].astype(int)
calculated['num_prior_admissions_12m'] = values['num_prior_admissions_12m'].astype(int).apply(lambda x: x ** 1.2)
calculated['num_active_medications'] = values['num_active_medications'].astype(int).apply(lambda x: x ** 1.5)
calculated['num_comorbidities'] = values['num_comorbidities'].astype(int).apply(lambda x: x ** 1.8)
calculated['systolic_bp'] = values['systolic_bp'].astype(float).apply(systolic_blood_pressure)
calculated['diastolic_bp'] = values['diastolic_bp'].astype(float).apply(diastolic_blood_pressure)
calculated['mean_arterial_pressure'] = values['mean_arterial_pressure'].astype(float).apply(map)
calculated['pulse_pressure'] = values['pulse_pressure'].astype(float).apply(pulse_pressure)
calculated['heart_rate'] = values['heart_rate'].astype(float).apply(lambda x: abs(x - 72) ** 4)
calculated['respiratory_rate'] = pd.array(respiratory_rate(values['age'].astype(float).tolist(), values['respiratory_rate'].astype(float).tolist()))
calculated['temperature_c'] = values['temperature_c'].astype(float).apply(temp)
calculated['spo2'] = values['spo2'].astype(float).apply(lambda x: (100 - x) ** 4)
calculated['gcs_total'] = values['gcs_total'].astype(int).apply(lambda x: (15 - x) ** 4)
calculated['pain_score'] = values['pain_score'].astype(int)
calculated['bmi'] = values['bmi'].astype(float).apply(bmi)
calculated['shock_index'] = values['shock_index'].astype(float)
calculated['news2_score'] = values['news2_score'].astype(int)

In [116]:
values

,arrival_mode,age_group,age,num_prior_ed_visits_12m,num_prior_admissions_12m,num_active_medications,num_comorbidities,systolic_bp,diastolic_bp,mean_arterial_pressure,pulse_pressure,heart_rate,respiratory_rate,temperature_c,spo2,gcs_total,pain_score,bmi,shock_index,news2_score
0,walk-in,young_adult,24,2,2,4,4,73.0,77.9,76.3,-4.9,75.3,26.7,38.5,88.8,10,10,27.7,1.032,13


In [117]:
fields

{'arrival_mode': 0.001325769307131233,
 'age_group': -0.005314758366319801,
 'age': -1.6605643438825475e-06,
 'num_prior_ed_visits_12m': 0.1257339657220237,
 'num_prior_admissions_12m': -0.0074174571486471386,
 'num_active_medications': 0.0001607885017735725,
 'num_comorbidities': 0.00022150971648483972,
 'systolic_bp': 6.091626949328199e-05,
 'diastolic_bp': 0.0004389954330244497,
 'mean_arterial_pressure': -0.0005314796352848906,
 'pulse_pressure': 3.2973311062823604e-09,
 'heart_rate': -2.3629554821677703e-09,
 'respiratory_rate': -2.0636271669141686e-07,
 'temperature_c': 0.0025212006386802813,
 'spo2': -5.24236405091187e-08,
 'gcs_total': 2.4831644753848334e-05,
 'pain_score': 0.06634725190450828,
 'bmi': 1.1776306607293347e-08,
 'shock_index': 0.037766820444874734,
 'news2_score': 0.13714166084207668}

In [118]:
calculated

,arrival_mode,age_group,age,num_prior_ed_visits_12m,num_prior_admissions_12m,num_active_medications,num_comorbidities,systolic_bp,diastolic_bp,mean_arterial_pressure,pulse_pressure,heart_rate,respiratory_rate,temperature_c,spo2,gcs_total,pain_score,bmi,shock_index,news2_score
0,0,1,121,2,2.297397,8.0,12.125733,289.0,0,0,4.064296e+06,118.5921,13107.9601,0.0625,15735.1936,625,10,53.1441,1.032,13


In [119]:
d = calculated.to_dict()
d = dict(zip(d.keys(), [f[0] for f in d.values()]))

calculatedParams = {}
for key in d:
    calculatedParams[key] = d[key] * fields[key]

In [120]:
# Show final result for priority score

wx = sum(calculatedParams.values())
b = intercept

print(wx, b, "\n")

print("Priority for given patient is:", np.round(wx + b))

2.7613270882232883 0.7208184578935732 

Priority for given patient is: 3.0
